# Context Model Training from Raw `.osz` (API)

This notebook trains only the context model using `train_api` (no CLI).
It keeps the same main settings as `train_context_raw_data.ipynb` and supports resume from `last.ckpt`.

In [ ]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)


In [ ]:
# Same intent/defaults as the CLI notebook
# Original raw-data lines (switch back later if needed):
# raw_osz_dir = Path("E:/batchbeatmapdownloadtesttemp")
# data_root = Path("E:/batchbeatmapdownloadtest")
raw_osz_dir = Path("sample_data_large/raw")
data_root = Path("sample_data_large")
repo_root = Path.cwd()
checkpoints_dir = repo_root / "checkpoints" / "sample_large_baseline"

epochs = 10
architecture_name = "taiko_transformer"
run_name = "test_10epochs"
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_baseline_largesample_data_api.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)
last_checkpoint = checkpoints_dir / "last.ckpt"

print(f"raw_osz_dir={raw_osz_dir}")
print(f"data_root={data_root}")
print(f"checkpoints_dir={checkpoints_dir}")
print(f"last_checkpoint={last_checkpoint}")
print(f"device={device}")


In [ ]:
# Prepare data artifacts from raw .osz inputs.
artifacts = prepare_sample_data_artifacts(
    osz_inputs=[str(raw_osz_dir)],
    data_root=data_root,
)
print(artifacts)

architecture_spec = ArchitectureSpec(name=architecture_name)
training_spec = TrainingSpec(
    epochs=epochs,
    device=device,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )


In [ ]:
if use_resume_if_available and last_checkpoint.exists():
    context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=device,
        checkpoints_dir=checkpoints_dir,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    context = create_training_context(
        data_root=data_root,
        architecture_spec=architecture_spec,
        training_spec=training_spec,
        checkpoints_dir=checkpoints_dir,
    )
    print("Starting a fresh baseline-model training run.")

print(f"start_epoch={context.start_epoch}")
print(f"target_epochs={epochs}")
print(f"architecture={context.architecture_spec.name}")


In [ ]:
context = train_context(
    context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
)

print("Training finished.")
print(f"last checkpoint: {(checkpoints_dir / 'last.ckpt').resolve()}")
print(f"best checkpoint: {(checkpoints_dir / 'best.ckpt').resolve()}")
